In [1]:
import cv2
import numpy as np
import json
import os

try:
    import pytesseract
    TESSERACT_AVAILABLE = True
except ImportError:
    TESSERACT_AVAILABLE = False

In [2]:
class ODLCPipeline:

    def __init__(self):
        print("Initializing ODLC Pipeline System...")

    # شغل ادهم هيكون هنا, مش هتغير البايبلاين هتعمل بس لود للتريند اوبجكت ديتيكش مودل هنا مكان ال نان

        self.detector = None
#-----------------------------------------

        self.shape_names = [
            "circle",
            "semicircle",
            "quarter circle",
            "triangle",
            "rectangle",
            "pentagon",
            "star",
            "cross"
        ]

        self.color_names = [
            "white",
            "black",
            "red",
            "blue",
            "green",
            "purple",
            "brown",
            "orange"
        ]


    #Main pipeline

    def run_pipeline(self, image_path):
        image = cv2.imread(image_path)

        if image is None:
            raise FileNotFoundError(
                f"Could not load image from: {image_path}"
            )

        height, width = image.shape[:2]

        print(
            f"Processing image: {image_path} "
            f"(Resolution: {width}x{height})"
        )

#هنا جزء ادهم برضو
#هتعدل فقط ال امبليمنتيشن بتاع detect_objects()
        detected_objects = self.detect_objects(image)

        final_results = []


#-------------------------------

        for obj in detected_objects:

            x_min, y_min, x_max, y_max = obj["bbox"]
            confidence = obj["confidence"]


            x_min = max(0, int(x_min))
            y_min = max(0, int(y_min))

            x_max = min(width, int(x_max))
            y_max = min(height, int(y_max))

            if x_min >= x_max or y_min >= y_max:
                print("Warning: Invalid bounding box. Skipping object.")
                continue

            cropped_target = image[y_min:y_max, x_min:x_max]

            if cropped_target.size == 0:
                print("Warning: Empty crop. Skipping object.")
                continue

#هنا جزء الكلاسيفيكيشن بتاعي

            classification = self.classify_object(cropped_target)

            center_x = (x_min + x_max) / 2
            center_y = (y_min + y_max) / 2

            pixel_coords = (center_x, center_y)

#هنا شغل هنا
#المفروض تبدلي فقط الامبلمنتيشن بتاع pixel_to_meters()

            ground_coords = self.pixel_to_meters(
                pixel_coords,
                width,
                height
            )


            target_report = {
                "confidence": round(float(confidence), 4),

                "bounding_box": {
                    "x_min": x_min,
                    "y_min": y_min,
                    "x_max": x_max,
                    "y_max": y_max
                },

                "pixel_center": {
                    "x": round(center_x, 2),
                    "y": round(center_y, 2)
                },

                "position_meters": {
                    "X": round(float(ground_coords[0]), 2),
                    "Y": round(float(ground_coords[1]), 2)
                },

                "classification": classification
            }
            final_results.append(target_report)

        return final_results



    def detect_objects(self, image):


#جزء أدهم
#ضيف كود الاوبجكت ديتكشن بتاعك هنا
        height, width = image.shape[:2]

        return [
            {
                "bbox": [
                    int(width * 0.35),
                    int(height * 0.30),
                    int(width * 0.65),
                    int(height * 0.60)
                ],
                "confidence": 0.95
            }
        ]

#------------------------------------------


    def classify_object(self, cropped_target):
        shape = self.detect_shape(cropped_target)
        shape_color = self.detect_shape_color(cropped_target)
        character = self.detect_character(cropped_target)
        character_color = self.detect_character_color(
            cropped_target,
            shape_color
        )

        return {
            "shape": shape,
            "shape_color": shape_color,
            "character": character,
            "character_color": character_color
        }



    def detect_shape(self, image):
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        gray = cv2.GaussianBlur(gray, (5, 5), 0)
        _, threshold = cv2.threshold(
            gray,
            0,
            255,
            cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )


        contours, _ = cv2.findContours(
            threshold,
            cv2.RETR_EXTERNAL,
            cv2.CHAIN_APPROX_SIMPLE
        )

        if not contours:
            return "unknown"

        contour = max(contours, key=cv2.contourArea)

        area = cv2.contourArea(contour)

        if area <= 0:
            return "unknown"

        perimeter = cv2.arcLength(contour, True)

        epsilon = 0.04 * perimeter

        approx = cv2.approxPolyDP(
            contour,
            epsilon,
            True
        )

        vertices = len(approx)

        x, y, w, h = cv2.boundingRect(contour)

        if w == 0 or h == 0:
            return "unknown"

        aspect_ratio = w / float(h)

        circularity = (
            4 * np.pi * area / (perimeter * perimeter)
            if perimeter > 0
            else 0
        )


        hull = cv2.convexHull(contour)

        hull_area = cv2.contourArea(hull)

        solidity = (
            area / hull_area
            if hull_area > 0
            else 0
        )



        # Circle
        if circularity > 0.80:
            return "circle"

        # Triangle
        if vertices == 3:
            return "triangle"

        # Rectangle
        if vertices == 4:

            # A normal rectangle has relatively high solidity.
            if solidity > 0.85:
                return "rectangle"

        # Pentagon
        if vertices == 5:
            return "pentagon"


        # Star

        if vertices >= 8 and solidity < 0.80:
            return "star"

        # Cross
        if vertices >= 8 and 0.80 <= solidity <= 0.95:
            return "cross"


        if 0.50 < circularity < 0.80:

            if aspect_ratio > 1.5 or aspect_ratio < 0.67:
                return "semicircle"

            return "quarter circle"

        return "unknown"





    def detect_shape_color(self, image):
        hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

        h = hsv[:, :, 0]
        s = hsv[:, :, 1]
        v = hsv[:, :, 2]


        valid_pixels = hsv.reshape(-1, 3)


        # White

        white_mask = (
            (s < 40) &
            (v > 180)
        )


        # Black

        black_mask = v < 50


        # Red

        red_mask = (
            ((h < 10) | (h > 170)) &
            (s > 80) &
            (v > 50)
        )


        # Orange

        orange_mask = (
            (h >= 10) &
            (h < 22) &
            (s > 80) &
            (v > 50)
        )


        # Yellow/Brown region

        brown_mask = (
            (h >= 8) &
            (h < 25) &
            (s > 80) &
            (v >= 30) &
            (v <= 180)
        )

        # Green

        green_mask = (
            (h >= 35) &
            (h < 85) &
            (s > 60) &
            (v > 40)
        )


        # Blue

        blue_mask = (
            (h >= 85) &
            (h < 135) &
            (s > 60) &
            (v > 40)
        )


        # Purple

        purple_mask = (
            (h >= 135) &
            (h < 170) &
            (s > 60) &
            (v > 40)
        )

        color_scores = {
            "white": int(np.sum(white_mask)),
            "black": int(np.sum(black_mask)),
            "red": int(np.sum(red_mask)),
            "orange": int(np.sum(orange_mask)),
            "brown": int(np.sum(brown_mask)),
            "green": int(np.sum(green_mask)),
            "blue": int(np.sum(blue_mask)),
            "purple": int(np.sum(purple_mask))
        }


        detected_color = max(
            color_scores,
            key=color_scores.get
        )

        if color_scores[detected_color] == 0:
            return "unknown"

        return detected_color


    def detect_character(self, image):
        if not TESSERACT_AVAILABLE:
            return "unknown"


        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)


        scale = 4

        enlarged = cv2.resize(
            gray,
            None,
            fx=scale,
            fy=scale,
            interpolation=cv2.INTER_CUBIC
        )


        enlarged = cv2.GaussianBlur(
            enlarged,
            (3, 3),
            0
        )


        _, binary = cv2.threshold(
            enlarged,
            0,
            255,
            cv2.THRESH_BINARY + cv2.THRESH_OTSU
        )


        config = (
            "--psm 10 "
            "-c tessedit_char_whitelist="
            "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"
        )

        text = pytesseract.image_to_string(
            binary,
            config=config
        )


        text = text.strip().upper()

        allowed = "ABCDEFGHIJKLMNOPQRSTUVWXYZ0123456789"

        valid_characters = [
            char
            for char in text
            if char in allowed
        ]

        if not valid_characters:
            return "unknown"


        return valid_characters[0]


    def detect_character_color(self, image, shape_color):
        hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

        h = hsv[:, :, 0]
        s = hsv[:, :, 1]
        v = hsv[:, :, 2]

        color_masks = {}


        # White

        color_masks["white"] = (
            (s < 50) &
            (v > 170)
        )


        # Black

        color_masks["black"] = v < 60


        # Red

        color_masks["red"] = (
            ((h < 10) | (h > 170)) &
            (s > 80) &
            (v > 50)
        )


        # Orange

        color_masks["orange"] = (
            (h >= 10) &
            (h < 22) &
            (s > 80) &
            (v > 50)
        )


        # Brown

        color_masks["brown"] = (
            (h >= 8) &
            (h < 25) &
            (s > 70) &
            (v >= 30) &
            (v <= 170)
        )


        # Green

        color_masks["green"] = (
            (h >= 35) &
            (h < 85) &
            (s > 60) &
            (v > 40)
        )


        # Blue

        color_masks["blue"] = (
            (h >= 85) &
            (h < 135) &
            (s > 60) &
            (v > 40)
        )


        # Purple

        color_masks["purple"] = (
            (h >= 135) &
            (h < 170) &
            (s > 60) &
            (v > 40)
        )



        scores = {}

        for color, mask in color_masks.items():

            score = int(np.sum(mask))


            if color == shape_color:
                score = 0

            scores[color] = score


        detected_color = max(
            scores,
            key=scores.get
        )

        if scores[detected_color] == 0:
            return "unknown"

        return detected_color


    def pixel_to_meters(
        self,
        pixel_coords,
        img_w,
        img_h
    ):

#جزء هنا
#ضيفي كود اللوكالايزيشن بتاعك هنا

        center_x = img_w / 2
        center_y = img_h / 2

        dx_pixels = pixel_coords[0] - center_x
        dy_pixels = pixel_coords[1] - center_y


#بالنسبة لهنا, غيري الرقم اللي جمب السكيل فاكتور هنا للالتيتيود جيومتري الصح

        scale_factor = 0.05

        x_meters = dx_pixels * scale_factor

        # Image Y increases downward.
        # Ground Y is therefore inverted.
        y_meters = -dy_pixels * scale_factor

        return (
            round(x_meters, 2),
            round(y_meters, 2)
        )



pipeline = ODLCPipeline()


# Put a test aerial image in your Colab Files and write its name here

image_path = "sample_drone_image.jpg"

try:

    results = pipeline.run_pipeline(image_path)

    print("\n========================================")
    print("         FINAL ODLC PIPELINE OUTPUT")
    print("========================================")

    print(
        json.dumps(
            results,
            indent=4
        )
    )

except FileNotFoundError as e:

    print(f"\nError: {e}")

except Exception as e:

    print(f"\nPipeline Error: {e}")

Initializing ODLC Pipeline System...

Error: Could not load image from: sample_drone_image.jpg
